# 01 · Data check (Week 1)

Goal: confirm the cleaned BTS data can answer the locked research question **before** any analysis.

> **Which airports or carriers show disproportionately high carrier-attributable delay, and does time-of-day or day-of-week make it worse?**

Prerequisite: `python scripts/download_bts.py && python scripts/build_dataset.py` (see README).

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from aviation_delays import data, metrics, charts

pd.set_option("display.float_format", "{:,.2f}".format)
DATA_PATH = Path(os.environ.get("AVIATION_DATA", data.PROCESSED_PATH))
flights = data.load_processed(DATA_PATH)
af = data.analysis_frame(flights)

## Cleaning summary
Rows are flagged, not dropped. Delay metrics use completed flights with a usable cause breakdown (`data.analysis_frame`).

In [ ]:
summary = data.cleaning_summary(flights)
summary["analysis_rows"] = len(af)
summary["analysis_share_of_rows"] = round(len(af) / len(flights), 4)
summary.to_frame("value")

## Answerability checks

| # | Check | Pass if |
|---|---|---|
| 1 | Cause fields are populated for delayed flights | < 2% of ≥15-min-late flights missing causes |
| 2 | Enough carriers to compare | ≥ 8 carriers, each ≥ 50k completed flights |
| 3 | Enough airports to compare | ≥ 30 origin airports with ≥ 20k completed flights |
| 4 | Time-of-day / day-of-week cells have volume | every hour 05–23 × weekday cell ≥ 1k flights |
| 5 | Carrier delay is a material share of cause-coded minutes | ≥ 15% (otherwise the question is not very interesting) |

In [ ]:
delayed = int(flights["arr_del15"].sum())
missing_share = flights["cause_missing"].sum() / delayed
carrier_n = af.groupby("carrier", observed=True).size().sort_values(ascending=False)
airport_n = af.groupby("origin", observed=True).size().sort_values(ascending=False)
cell_n = af[af["dep_hour"].between(5, 23)].groupby(["dep_hour", "day_of_week"], observed=True).size()
carrier_share = metrics.network_summary(af)["carrier_share"]

checks = pd.DataFrame(
    [
        ("1 cause fields populated", f"{missing_share:.2%} of delayed flights missing causes", missing_share < 0.02),
        ("2 carriers", f"{(carrier_n >= 50_000).sum()} carriers with >= 50k flights", (carrier_n >= 50_000).sum() >= 8),
        ("3 airports", f"{(airport_n >= 20_000).sum()} airports with >= 20k flights", (airport_n >= 20_000).sum() >= 30),
        ("4 time cells", f"smallest hour x weekday cell: {cell_n.min():,} flights", cell_n.min() >= 1_000),
        ("5 carrier share", f"carrier = {carrier_share:.1%} of cause-coded minutes", carrier_share >= 0.15),
    ],
    columns=["check", "result", "pass"],
)
checks

In [ ]:
# Volume per carrier (for the min-volume threshold used in 02_analysis)
carrier_n.to_frame("completed_flights")

In [ ]:
# Cause coverage by month: a sudden gap would mean a reporting problem in that month
flights.assign(month=flights["date"].dt.strftime("%Y-%m")).groupby("month").agg(
    flights=("status", "size"),
    cancelled_rate=("status", lambda s: (s == "cancelled").mean()),
    delayed_15=("arr_del15", "sum"),
    cause_missing=("cause_missing", "sum"),
)

## Decision

*Fill in after running on the real data.*

- Answerable as locked? **yes / no**
- If no: which check failed, and why. **Flag it back. Don't quietly swap in a different question.**
- Thresholds carried into 02: carriers ≥ 50k flights, airports ≥ 20k flights (adjust here if the checks suggest otherwise).